In [1]:
import sys
sys.path.append('../')

%env MUJOCO_GL=egl

env: MUJOCO_GL=egl


In [13]:
import jax
import jax.numpy as jnp
import mujoco
import numpy as np
import mediapy as media
from dataclasses import dataclass, field
from mujoco.mjx._src import math as mjx_math

from builderbench.env_utils import make_env
from utils.wrapper import wrap_env

In [3]:
@dataclass
class Args:
    # experiment
    agent: str = "mp"
    seed: int = 1

    # environment
    env_id: str = 'creative-1-task1'
    env_early_termination: bool = True
    env_episode_length: int = None
    permutation_invariant_reward: bool = True   # invariance to the order of cubes in any structure

    # planner
    kp_pos: float = 10.0
    kd_pos: float = 2.0

    kp_yaw: float = 1.0
    kd_yaw: float = 0.5
    
    num_envs: str = 1
    

In [4]:
args = Args()

In [5]:
env_class, default_config = make_env(args)
env = env_class(config=default_config)

Warp 1.9.0 initialized:
   CUDA Toolkit 12.8, Driver 13.0
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA A100-SXM4-80GB" (79 GiB, sm_80, mempool enabled)
     "cuda:1"   : "NVIDIA A100-SXM4-80GB" (79 GiB, sm_80, mempool enabled)
     "cuda:2"   : "NVIDIA A100-SXM4-80GB" (79 GiB, sm_80, mempool enabled)
     "cuda:3"   : "NVIDIA A100-SXM4-80GB" (79 GiB, sm_80, mempool enabled)
     "cuda:4"   : "NVIDIA A100-SXM4-80GB" (79 GiB, sm_80, mempool enabled)
     "cuda:5"   : "NVIDIA A100-SXM4-80GB" (79 GiB, sm_80, mempool enabled)
     "cuda:6"   : "NVIDIA A100-SXM4-80GB" (79 GiB, sm_80, mempool enabled)
     "cuda:7"   : "NVIDIA A100-SXM4-80GB" (79 GiB, sm_80, mempool enabled)
   CUDA peer access:
     Supported fully (all-directional)
   Kernel cache:
     /home/nvidia/.cache/warp/1.9.0


In [6]:
np.random.seed(args.seed)
key = jax.random.PRNGKey(args.seed)
key, key_env, key_eval, key_policy, key_value = jax.random.split(key, 5)

In [7]:
reset_fn = jax.jit(env.reset)
step_fn = jax.jit(env.step)

In [8]:
@jax.jit
def get_yaw_from_quat(q):
    w, x, y, z = q[0], q[1], q[2], q[3]
    siny_cosp = 2 * (w * z + x * y)
    cosy_cosp = 1 - 2 * (y * y + z * z)
    yaw = jnp.arctan2(siny_cosp, cosy_cosp)
    return yaw

@jax.jit
def normalize_angle(angle):
    return jnp.arctan2(jnp.sin(angle), jnp.cos(angle))

In [9]:
@jax.jit
def get_waypoint(env_state):
    current_pos = env_state.obs[:3*env._config.num_cubes].reshape(env._config.num_cubes, 3)[cube_id]
    current_quat = env_state.obs[3*env._config.num_cubes:][:4*env._config.num_cubes].reshape(env._config.num_cubes, 4)[cube_id]
    current_linvel = env_state.obs[7*env._config.num_cubes:][:3*env._config.num_cubes].reshape(env._config.num_cubes, 3)[cube_id]
    current_angvel = env_state.obs[10*env._config.num_cubes:][:3*env._config.num_cubes].reshape(env._config.num_cubes, 3)[cube_id]
    
    target_pos = env_state.info['target_goal']
    
    current_xy = current_pos[:2]
    target_xy = target_pos[:2]
    
    current_height = current_pos[-1]
    top_height = target_pos[-1] + 0.1
    
    horizontal_dist_to_target = jnp.linalg.norm(current_xy - target_xy)

    is_far = horizontal_dist_to_target > 0.01
    is_low = current_height < (top_height - 0.005)

    wp_lift = target_pos.at[:2].set(current_xy).at[2].set(top_height)
    wp_hover = target_pos.at[:2].set(target_xy).at[2].set(top_height)
    wp_final = target_pos
        
    current_waypoint = jnp.where(
        is_far,
        jnp.where(is_low, wp_lift, wp_hover),
        wp_final
    )

    return current_waypoint


In [61]:
@jax.jit
def get_action(env_state, waypoint, cube_id):
    current_pos = env_state.obs[:3*env._config.num_cubes].reshape(env._config.num_cubes, 3)[cube_id]
    current_quat = env_state.obs[3*env._config.num_cubes:][:4*env._config.num_cubes].reshape(env._config.num_cubes, 4)[cube_id]
    current_linvel = env_state.obs[7*env._config.num_cubes:][:3*env._config.num_cubes].reshape(env._config.num_cubes, 3)[cube_id]
    current_angvel = env_state.obs[10*env._config.num_cubes:][:3*env._config.num_cubes].reshape(env._config.num_cubes, 3)[cube_id]
    
    current_yaw = get_yaw_from_quat(current_quat)
    
    error_pos = waypoint - current_pos
    output_pos = (args.kp_pos * error_pos) + (args.kd_pos * - current_linvel)
    output_pos.at[2].set( output_pos[2] + gravity_comp )
    
    error_yaw = normalize_angle(0.0 - current_yaw)
    output_yaw = (args.kp_yaw * error_yaw) + (args.kd_yaw * - current_angvel[-1])

    ctrl_action = jnp.concatenate([output_pos, output_yaw[None]], axis=0)
    # ctrl_action = jnp.clip(ctrl_action, env._ctrl_bounds[0][:4], env._ctrl_bounds[1][:4])
    ctrl_action = ( ctrl_action - env._ctrl_median[:4] ) / env._ctrl_halfspan[:4]
    
    select_action = ( ( ( 2 * cube_id + 1) * jnp.pi / 3 ) - jnp.pi ) / ( jnp.pi )

    action =  jnp.concatenate([ctrl_action, select_action[None]], axis=0)
    # action = jnp.clip(action, -1, 1)
    
    return action, output_pos

In [62]:
env._ctrl_bounds[0][:4]

Array([-0.1, -0.1,  0. , -0.1], dtype=float32)

In [63]:
cube_id  = 0
cube_mass = 0.004188
gravity = 9.81
gravity_comp = cube_mass * gravity

In [64]:
camera = mujoco.MjvCamera()
camera.distance = 0.8
camera.lookat = np.array([0.4, 0.0 , 0.4])
camera.elevation = -30.0
camera.azimuth = 180

In [65]:
rollout = []

env_state = reset_fn(key_env)
rollout.append(env_state)

for i in range(default_config.episode_length):
    wp = get_waypoint(env_state)
    action, out_wp = get_action(env_state, wp, cube_id)
    print(wp, out_wp)
    env_state = step_fn(env_state, action)
    rollout.append(env_state)

[0.06075218 0.         0.12      ] [0.        0.        1.0000001]
[0.06075218 0.         0.12      ] [0.        0.        0.8846453]
[0.06075218 0.         0.12      ] [0.        0.        0.8213438]
[0.06075218 0.         0.12      ] [0.        0.        0.7873622]
[0.06075218 0.         0.12      ] [0.        0.        0.7698136]
[0.06075218 0.         0.12      ] [0.        0.        0.7614018]
[0.06075218 0.         0.12      ] [0.       0.       0.758004]
[0.06075218 0.         0.12      ] [0.        0.        0.7572983]
[0.06075218 0.         0.12      ] [0.       0.       0.757983]
[0.06075218 0.         0.12      ] [0.         0.         0.75933456]
[0.06075218 0.         0.12      ] [0.        0.        0.7609557]
[0.06075218 0.         0.12      ] [0.         0.         0.76263297]
[0.06075218 0.         0.12      ] [0.        0.        0.7642567]
[0.06075218 0.         0.12      ] [0.         0.         0.76577353]
[0.06075218 0.         0.12      ] [0.        0.        0.7

In [66]:
video_images = []
mocap_key = 'target_mocap'
for i in range(default_config.episode_length):
    if i % 2 == 0:
        video_images.append(
            env.render_from_info(
                rollout[i].data.qpos, 
                rollout[i].data.qvel, 
                rollout[i].info[f'{mocap_key}_pos'],
                rollout[i].info[f'{mocap_key}_quat'],
                camera=camera,
            )
        )

In [68]:
media.show_video(video_images, fps=1.0 / env.dt / 2)

In [51]:
env.mj_model.actuator_ctrlrange

array([[-0.1,  0.1],
       [-0.1,  0.1],
       [ 0. ,  1. ],
       [-0.1,  0.1]])